
# 04 · LLM Agents — zero-shot → agentic → multi-agent

**Vai trò notebook**: 3 LLM patterns theo độ phức tạp tăng dần. Multi-agent là headline.

**Owner**: Duc
**Deadline**: 2026-05-22
**Slide chapter**: 4 — Implementation (Strategies → LLM subsection)

## Mục tiêu
1. **Zero-shot**: 1 prompt → 1 response → weights. Baseline LLM.
2. **Single-agentic**: ReAct loop với 4 tools (price, indicators, news, fundamentals).
3. **Multi-agent**: 8-role LangGraph (3 analysts → bull/bear 2 rounds → trader → risk → portfolio).
4. Cost / latency / accuracy comparison.

## Defense Q&A
- Q: Tại sao 3 patterns? Multi-agent có cần thiết không?
- Q: Multi-agent 8 roles — vai trò mỗi role?
- Q: Cost LLM tổng cộng bao nhiêu? Có affordable không?
- Q: Cache prompt-response — cơ chế thế nào?
- Q: Multi-agent fallback khi node error?


## Setup


In [ ]:
import sys
from pathlib import Path

_NB_DIR = Path().resolve()
if _NB_DIR.name != "notebooks":
    _NB_DIR = _NB_DIR / "notebooks"
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from _shared import (  # noqa: E402
    AGENT_COLORS,
    BASELINES,
    DATA,
    FIGURES,
    LLM_AGENTS,
    RESULTS,
    RL_AGENTS,
    ROLE_COLORS,
    TRANSCRIPTS,
    assert_frozen_snapshot,
    list_transcript_dates,
    load_curve,
    load_holdings,
    load_metrics_json,
    load_metrics_table,
    load_transcript,
    save_fig,
    setup_matplotlib,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

setup_matplotlib()
assert_frozen_snapshot()
metrics = load_metrics_table()
print("snapshot OK · agents:", list(metrics.index))



## TODO-01: model-lock-summary
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `src/llm/client.py:OpenAIClient`
- **WRITE**: markdown explaining model lock + table
- **CONSTRAINTS**:
  - Bảng: agent → primary model → reason
    | Agent | Model | Why |
    | --- | --- | --- |
    | zero_shot | gpt-4o-mini | Cheap baseline |
    | single_agentic | gpt-4o | Tool calling capable |
    | multi_agent (analysts) | gpt-4o-mini | High volume, simple task |
    | multi_agent (researchers/trader/risk/PM) | gpt-4o | Deliberation needs depth |
  - Note: cutoff Oct 2023 → test window 2025-05+ là OOD → no leakage
- **DEFENSE Q&A**: "Chọn model nào? Tại sao?"


In [ ]:
# TODO-01: model lock table
from src.llm.client import OpenAIClient
print(OpenAIClient.ALLOWED_MODELS)



## TODO-02: zero-shot-walkthrough
- **OWNER**: Duc   **DEPENDS**: TODO-01
- **READ**: `src/llm/zero_shot.py`, `src/llm/serialize.py`
- **WRITE**:
  - Markdown: prompt structure (system + user message), no tools, single call
  - Code cell: in ra prompt template (10-20 dòng đầu) + 1 cached response từ `results/zero_shot/decisions.jsonl`
  - Code cell: parse response → in weights dict
- **CONSTRAINTS**:
  - Show 1 sample (date, prompt excerpt, response excerpt, parsed_weights)
  - Highlight parser robustness (fallback to hold on JSON fail)
- **VALIDATE**: parsed weights sum ≤ 1, length 5
- **PATTERN**: `src/llm/parser.py:parse_weights`


In [ ]:
# TODO-02: zero-shot prompt + 1 cached decision
import json
sample = json.loads(open(RESULTS / 'zero_shot' / 'decisions.jsonl').readline())
print(sample)



## TODO-03: single-agentic-walkthrough
- **OWNER**: Duc   **DEPENDS**: TODO-02
- **READ**: `src/llm/single_agentic.py`, `src/llm/tools.py`
- **WRITE**:
  - Markdown: 4 tools (get_price, get_indicators, get_news, get_fundamentals), ReAct loop (think → call → result → think)
  - `report/figures/04__single_agentic_loop.png` — diagram cycle
  - Code cell: in 1 cached transcript showing tool-call sequence
- **CONSTRAINTS**:
  - Max iterations = 5 (config)
  - Tool spec: name, args, return type
- **VALIDATE**: sample transcript có ≥ 1 tool call
- **PATTERN**: `results/single_agentic/decisions.jsonl` chứa cached transcripts


In [ ]:
# TODO-03: single_agentic loop + transcript sample
pass



## TODO-04: multi-agent-topology
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `src/llm/multi_agent/graph.py`, `src/llm/multi_agent/state.py`
- **WRITE**: `report/figures/04__multi_agent_topology.png` + bảng 8 roles
- **CONSTRAINTS**:
  - 8 nodes: technical / news / fundamental analyst (col 1) → bullish + bearish researcher (col 2, debate 2 rounds) → trader (col 3) → risk_manager (col 4) → portfolio_manager (col 5)
  - Colors = ROLE_COLORS
  - Table 8 rows × (role, model, primary output field, downstream consumer)
- **VALIDATE**: 8 nodes; all edges match `graph.py`
- **PATTERN**: tham khảo `frontend/components/DebateGraph.tsx` cho positions
- **DEFENSE Q&A** (critical): "Vẽ kiến trúc multi-agent?" → mở figure này.


In [ ]:
# TODO-04: multi-agent topology diagram
pass



## TODO-05: multi-agent-decision-replay — 1 full decision
- **OWNER**: Duc   **DEPENDS**: TODO-04
- **READ**: `results/multi_agent/transcripts/2026-04-28.json`
- **WRITE**: 10 markdown cells walking through transcript entries
- **CONSTRAINTS**:
  Mỗi entry render dạng:
  ```
  ### technical_analyst (gpt-4o-mini, 11:12:22)
  > ## VCB
  > Setup: sideways trong 10 phiên... RSI z-score +0.89...
  > Nhận định: **neutral** — chưa có tín hiệu mạnh.
  > (continue cho 4 ticker còn lại)
  ```
  - Mỗi entry truncate 300 chars để không quá dài
- **VALIDATE**: 10 cells render (3 analysts + 2 bull + 2 bear + trader + risk + portfolio)
- **DEFENSE Q&A**: "Cho xem 1 quyết định multi-agent cụ thể?" → walk qua notebook này


In [ ]:
# TODO-05: replay 2026-04-28 transcript
transcript = load_transcript('2026-04-28')
for entry in transcript['transcript']:
    print(f"### {entry['role']} ({entry.get('model')}, {entry.get('ts')})")
    print('>', entry.get('output', '')[:300])
    print()



## TODO-06: debate-rounds-distribution
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `results/multi_agent/decisions.jsonl` (51 decisions)
- **WRITE**: `report/figures/04__debate_rounds_hist.png`
- **CONSTRAINTS**:
  - X = debate_rounds, Y = count
  - Tất cả 51 decisions: debate_rounds = 2 (cap PRD §7)
  - Note trên figure: "Bull/bear debate cap = 2 vòng theo PRD §7"
- **VALIDATE**:
  ```python
  decisions = [json.loads(l) for l in open(RESULTS / 'multi_agent' / 'decisions.jsonl')]
  rounds = [d['debate_rounds'] for d in decisions]
  assert max(rounds) <= 2
  ```
- **DEFENSE Q&A**: "Debate có converge không? Bao nhiêu rounds?" → constant 2 rounds (deterministic cap, không adaptive).


In [ ]:
# TODO-06: debate rounds histogram
pass



## TODO-07: cost-latency-table
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `metrics` cho 3 LLM agents
- **WRITE**: bảng so sánh + `report/figures/04__llm_cost_latency.png`
- **CONSTRAINTS**:
  Bảng:
  | Agent | n_decisions | llm_cost_usd | avg_latency_s | total cost if scaled to 248 days |
  |---|---|---|---|---|
  | zero_shot | 10 | ? | ? | ? × 24.8 |
  | single | 10 | ? | ? | ? × 24.8 |
  | multi | 51 | $3.21 | ~35s | (already weekly cadence) |
  Hint: zero_shot/single là smoke; multi full.
- **VALIDATE**: numbers từ `metrics.loc[agent, 'llm_cost_usd']`, không hardcode
- **DEFENSE Q&A**: "Cost tổng cộng?" → multi $3.21 full; zero/single smoke. Full LLM run ~$15 total.


In [ ]:
# TODO-07: cost + latency comparison
llm_view = metrics.loc[list(LLM_AGENTS)][['n_decisions','llm_cost_usd','avg_latency_s']]
print(llm_view.to_markdown())



## TODO-08: cache-mechanism
- **OWNER**: Duc   **DEPENDS**: none
- **READ**: `src/llm/client.py` cache logic, env var `LLM_CACHE_PATH`
- **WRITE**: markdown explanation + show cache hit example
- **CONSTRAINTS**:
  - Cache key = `(date, ticker_set, prompt_hash)` → unique per decision
  - Cache hit ⇒ 0 LLM call ⇒ free rerun
  - File-based (JSON lines hoặc SQLite — check code)
- **DEFENSE Q&A**: "Reruns notebooks tốn LLM cost không?" → KHÔNG; cache hits 100% sau lần backtest đầu.


In [ ]:
# TODO-08: cache mechanism explanation
pass



## Defense Q&A — câu trả lời sẵn

> **Q1: Tại sao 3 patterns LLM?**
> A: Hierarchy theo độ phức tạp. Zero-shot = baseline (pretrained knowledge only). Single-agentic = thêm tool access (price/news/indicators). Multi-agent = thêm deliberation (debate, risk review). Mỗi tier kế thừa tier dưới + thêm capability → cho phép isolate effect của từng feature.
> Evidence: cum_return order multi (50%) > single (6.5%) > zero (8.7%) — single < zero do smoke N=10 không representative.

> **Q2: 8 roles multi-agent?**
> A: TODO-04 topology. 3 analysts cung cấp signal độc lập (kỹ thuật, tin, fundamental). Bull/bear debate 2 vòng stress-test conviction. Trader tổng hợp. Risk review concentration. Portfolio finalize weights.

> **Q3: Total LLM cost?**
> A: Multi-agent full backtest = $3.21 (51 weekly decisions × ~$0.06 each). Zero/single smoke ~ $0.30 each. Tổng ~ $4 cho toàn bộ thesis.
> Evidence: TODO-07 + `results/multi_agent/decisions.jsonl` cost_delta_usd field.

> **Q4: Cache prompt-response?**
> A: File-based, key = (date, ticker_set, prompt_hash). Cache hit = 100% sau lần backtest đầu → notebook reruns free. Cache cũng cho phép defense demo (mở /debate UI lấy transcript) mà không cần internet/OpenAI.
> Evidence: TODO-08 + `src/llm/client.py:OpenAIClient.complete` cache check.

> **Q5: Multi-agent fallback khi node error?**
> A: Mỗi node có timeout 60s. Timeout / error → log + fallback to hold action (target_weights = previous). 0 errors qua 51 decisions của PKG-S → không cần fallback path nào thực sự fire.
> Evidence: `results/multi_agent/metrics.json` `node_errors_total: 0`, `timeout_rate: 0.0`.
